# Corn Futures vs Weather: EDA and Backtest

Explore the relationship between Corn Belt precipitation and corn futures prices,
build a simple long/short signal based on 30-day rolling precipitation, and
backtest it over the last year of data.

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from signal_gen import (
    load_futures, load_weather, merge_on_trading_days,
    add_avg_precip, add_rolling_precip_lag, generate_signal,
)
from backtest import run_backtest

logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(asctime)s %(name)s %(levelname)s %(message)s")
logger = logging.getLogger("eda")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Load and Merge Data

In [ ]:
futures = load_futures()
weather = load_weather()
df = merge_on_trading_days(futures, weather)
df = add_avg_precip(df)

logger.info("Date range: %s to %s", df.index[0].date(), df.index[-1].date())
logger.info("Shape: %s", df.shape)
df.head()

## 2. EDA: Corn Futures

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price time series
axes[0].plot(df.index, df["Close"], linewidth=0.8)
axes[0].set_title("Corn Futures Close Price")
axes[0].set_ylabel("Price (cents/bushel)")
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Daily returns distribution
daily_returns = df["Close"].pct_change().dropna()
axes[1].hist(daily_returns, bins=100, edgecolor="black", linewidth=0.3)
axes[1].set_title("Daily Return Distribution")
axes[1].set_xlabel("Daily Return")
axes[1].set_ylabel("Frequency")

# Volume over time
axes[2].bar(df.index, df["Volume"], width=2, alpha=0.6)
axes[2].set_title("Trading Volume")
axes[2].set_ylabel("Volume")
axes[2].xaxis.set_major_locator(mdates.YearLocator())
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

logger.info("Return stats -- mean: %.4f, std: %.4f, min: %.4f, max: %.4f",
            daily_returns.mean(), daily_returns.std(), daily_returns.min(), daily_returns.max())

## 3. EDA: Weather (Precipitation)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Daily avg precipitation with 30-day rolling mean
axes[0].bar(df.index, df["avg_precip_in"], width=2, alpha=0.3, label="Daily avg")
rolling_30 = df["avg_precip_in"].rolling(30).mean()
axes[0].plot(df.index, rolling_30, color="red", linewidth=1, label="30-day rolling mean")
axes[0].set_title("Average Corn Belt Precipitation")
axes[0].set_ylabel("Precipitation (inches)")
axes[0].legend()
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Monthly avg precipitation by year (heatmap)
df_monthly = df["avg_precip_in"].groupby([df.index.year, df.index.month]).mean().unstack()
im = axes[1].imshow(df_monthly.values, aspect="auto", cmap="Blues")
axes[1].set_title("Monthly Avg Precipitation by Year")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Year")
axes[1].set_yticks(range(len(df_monthly.index)))
axes[1].set_yticklabels(df_monthly.index)
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.colorbar(im, ax=axes[1], label="Inches")

plt.tight_layout()
plt.show()

## 4. Weather vs Futures Relationship

In [ ]:
# Add lagged rolling precip (no signal yet, just the feature)
df = add_rolling_precip_lag(df, window=30, lag=1)

# Forward returns for correlation analysis
df["fwd_1d_return"] = df["Close"].pct_change().shift(-1)
df["fwd_5d_return"] = df["Close"].pct_change(5).shift(-5)

valid = df.dropna(subset=["rolling_precip", "fwd_1d_return"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: lagged 30-day precip vs next-day return
axes[0].scatter(valid["rolling_precip"], valid["fwd_1d_return"], alpha=0.1, s=5)
axes[0].set_title("30-Day Lagged Precip vs Next-Day Return")
axes[0].set_xlabel("Rolling 30-day precip sum (inches)")
axes[0].set_ylabel("Next-day return")

# Correlation by month
monthly_corr = valid.groupby(valid.index.month).apply(
    lambda g: g["rolling_precip"].corr(g["fwd_1d_return"])
)
axes[1].bar(monthly_corr.index, monthly_corr.values)
axes[1].set_title("Precip-Return Correlation by Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Correlation")
axes[1].set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

overall_corr = valid["rolling_precip"].corr(valid["fwd_1d_return"])
corr_5d = valid.dropna(subset=["fwd_5d_return"])["rolling_precip"].corr(
    valid.dropna(subset=["fwd_5d_return"])["fwd_5d_return"]
)
logger.info("Overall correlation (1d): %.4f, (5d): %.4f", overall_corr, corr_5d)

## 5. Signal Generation

Calibrate thresholds using the rolling precipitation distribution, then generate
the long/short signal. We use the 75th percentile as the long threshold and the
25th percentile as the short threshold.

In [ ]:
# Use training period (before 2024-03-10) to calibrate thresholds
TRAIN_END = "2024-03-10"

rolling_train = df.loc[:TRAIN_END, "rolling_precip"].dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rolling_train, bins=80, edgecolor="black", linewidth=0.3)
ax.set_title("Distribution of 30-Day Rolling Precip (Training Period)")
ax.set_xlabel("Rolling 30-day precip sum (inches)")
ax.set_ylabel("Frequency")

threshold_long = rolling_train.quantile(0.75)
threshold_short = rolling_train.quantile(0.25)
ax.axvline(threshold_long, color="green", linestyle="--", label=f"Long >= {threshold_long:.2f}")
ax.axvline(threshold_short, color="red", linestyle="--", label=f"Short <= {threshold_short:.2f}")
ax.legend()

plt.tight_layout()
plt.show()

logger.info("Thresholds -- long: %.2f, short: %.2f", threshold_long, threshold_short)

In [ ]:
# Generate signal on the full dataset using training-calibrated thresholds
df = generate_signal(df, threshold_long, threshold_short)

# Signal overlaid on price chart
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(df.index, df["Close"], linewidth=0.8, color="black", alpha=0.7)
ax1.set_ylabel("Close Price (cents/bushel)")
ax1.set_title("Corn Futures with Signal Overlay")

long_mask = df["signal"] == 1
short_mask = df["signal"] == -1
ax1.fill_between(df.index, df["Close"].min(), df["Close"].max(),
                 where=long_mask, alpha=0.15, color="green", label="Long")
ax1.fill_between(df.index, df["Close"].min(), df["Close"].max(),
                 where=short_mask, alpha=0.15, color="red", label="Short")
ax1.legend()
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

logger.info("Signal counts -- long: %d, short: %d, flat: %d",
            long_mask.sum(), short_mask.sum(), (df["signal"] == 0).sum())

## 6. Backtest

Run the backtest on the **out-of-sample period** (2024-03-10 onward) using
thresholds calibrated on the training set. This prevents overfitting the
signal to the evaluation period.

In [ ]:
# Backtest on out-of-sample period only
df_oos = df.loc[TRAIN_END:]
backtest_df, trade_log, stats = run_backtest(df_oos, cost_per_trade=0.0)

# Cumulative P&L curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(backtest_df.index, backtest_df["cumulative_pnl"], linewidth=1.2)
axes[0].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title("Cumulative P&L (Out-of-Sample)")
axes[0].set_ylabel("P&L (cents/bushel)")
axes[0].xaxis.set_major_locator(mdates.MonthLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

# Drawdown
running_max = backtest_df["cumulative_pnl"].cummax()
drawdown = backtest_df["cumulative_pnl"] - running_max
axes[1].fill_between(backtest_df.index, drawdown, 0, color="red", alpha=0.4)
axes[1].set_title("Drawdown")
axes[1].set_ylabel("Drawdown (cents/bushel)")
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
stats_df = pd.DataFrame([
    ("Total P&L (cents/bushel)", f"{stats['total_pnl']:.2f}"),
    ("Number of Trades", f"{stats['num_trades']}"),
    ("Win Rate", f"{stats['win_rate']:.1%}"),
    ("Avg Win", f"{stats['avg_win']:.2f}"),
    ("Avg Loss", f"{stats['avg_loss']:.2f}"),
    ("Max Drawdown", f"{stats['max_drawdown']:.2f}"),
    ("Sharpe Ratio (annualized)", f"{stats['sharpe_ratio']:.2f}"),
    ("Avg Holding Period (days)", f"{stats['avg_holding_days']:.1f}"),
], columns=["Metric", "Value"])

stats_df

In [ ]:
# Trade log
logger.info("Showing first 20 of %d trades", len(trade_log))
trade_log.head(20)

## 7. Discussion

**Signal**: Long when 30-day rolling Corn Belt precipitation exceeds the 75th
percentile (training set); short when below the 25th percentile. The economic
logic is that heavy rain damages corn crops, reducing supply and pushing
futures prices higher.

**Limitations**:
- No transaction costs (configurable via `cost_per_trade` parameter)
- Single contract, no position sizing
- Year-round trading; seasonal filtering (May-Sep) could improve signal quality
- Simple univariate signal -- temperature, soil moisture, USDA reports not used
- Thresholds chosen at fixed quantiles; walk-forward optimization could help

**Next steps**:
- Filter signal to growing season only (May-Sep)
- Add temperature extremes as a second factor
- Walk-forward threshold optimization
- Include transaction cost sensitivity analysis